# Exp: Effect of excluding post-MPM 5-day window

**Question**: Does excluding the 5 days after each meeting actually improve model performance?

| Model | Training rows | Test rows |
|-------|--------------|----------|
| **A (main)** | `is_post_mpm == 0` only | `is_post_mpm == 0` only |
| **B (no filter)** | all rows | all rows |

**Implementation**: Model B is run by zeroing out `is_post_mpm` in a copy of `df_fly`,  
so `get_features_and_target` in `src/modeling.py` includes all rows without code changes.

**Metrics to compare**: Global IC, CS IC, Train IC, Gap — for both 3d and 5d horizons.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.stats import spearmanr

from src.processing        import load_and_clean_data
from src.features_rv       import generate_rv_features
from src.pooling_butterfly import pool_butterfly_data
from src.modeling          import walk_forward_with_model, summarize_ic

EXCEL_PATH   = '../data/BOJ_data.xlsx'
MEETING_PATH = '../data/BOJ_meeting_history.csv'
START_DATE   = '2024-01-01'
FLY_INDICES  = set(range(2, 8))   # B2-B7

print('Loading data...')
df_raw = load_and_clean_data(EXCEL_PATH, MEETING_PATH)
df_rv  = generate_rv_features(df_raw)
df_fly = pool_butterfly_data(df_rv)

n_post  = (df_fly['is_post_mpm'] == 1).sum()
n_total = len(df_fly)
print(f'Total rows : {n_total}')
print(f'Post-MPM   : {n_post}  ({100 * n_post / n_total:.1f}%)')
print(f'Normal     : {n_total - n_post}  ({100 * (n_total - n_post) / n_total:.1f}%)')

## Model A — Main (post-MPM excluded)

In [ ]:
print('Running Model A (main, post-MPM excluded)...')
res_a_3d, *_ = walk_forward_with_model(df_fly, 'Target_3d_norm', START_DATE)
res_a_5d, *_ = walk_forward_with_model(df_fly, 'Target_5d_norm', START_DATE)

ic_a_3d = summarize_ic(res_a_3d, instrument_indices=FLY_INDICES)
ic_a_5d = summarize_ic(res_a_5d, instrument_indices=FLY_INDICES)

print(f"Model A 3d  Global={ic_a_3d['ic_all']:.3f}  CS={ic_a_3d['cs_ic']:.3f}  "
      f"Train={ic_a_3d['train_ic']:.3f}  Gap={ic_a_3d['gap']:.3f}")
print(f"Model A 5d  Global={ic_a_5d['ic_all']:.3f}  CS={ic_a_5d['cs_ic']:.3f}  "
      f"Train={ic_a_5d['train_ic']:.3f}  Gap={ic_a_5d['gap']:.3f}")

## Model B — No filter (post-MPM included)

`is_post_mpm` を全行 0 に上書きしたコピーを渡すことで、
`get_features_and_target` のフィルタを無効化する。コード変更なし。

In [ ]:
df_fly_all = df_fly.copy()
df_fly_all['is_post_mpm'] = 0   # disable the filter in get_features_and_target

print('Running Model B (no filter, post-MPM included)...')
res_b_3d, *_ = walk_forward_with_model(df_fly_all, 'Target_3d_norm', START_DATE)
res_b_5d, *_ = walk_forward_with_model(df_fly_all, 'Target_5d_norm', START_DATE)

ic_b_3d = summarize_ic(res_b_3d, instrument_indices=FLY_INDICES)
ic_b_5d = summarize_ic(res_b_5d, instrument_indices=FLY_INDICES)

print(f"Model B 3d  Global={ic_b_3d['ic_all']:.3f}  CS={ic_b_3d['cs_ic']:.3f}  "
      f"Train={ic_b_3d['train_ic']:.3f}  Gap={ic_b_3d['gap']:.3f}")
print(f"Model B 5d  Global={ic_b_5d['ic_all']:.3f}  CS={ic_b_5d['cs_ic']:.3f}  "
      f"Train={ic_b_5d['train_ic']:.3f}  Gap={ic_b_5d['gap']:.3f}")

## Comparison table

In [ ]:
rows = [
    ('A  post-MPM excluded  3d', ic_a_3d, len(res_a_3d)),
    ('B  post-MPM included  3d', ic_b_3d, len(res_b_3d)),
    ('A  post-MPM excluded  5d', ic_a_5d, len(res_a_5d)),
    ('B  post-MPM included  5d', ic_b_5d, len(res_b_5d)),
]

hdr = f"{'Model':<30}  {'Global IC':>9}  {'CS IC':>7}  {'Train IC':>9}  {'Gap':>7}  {'n_obs':>6}"
print(hdr)
print('-' * len(hdr))
for label, ic, n in rows:
    print(f"{label:<30}  {ic['ic_all']:>9.3f}  {ic['cs_ic']:>7.3f}  "
          f"{ic['train_ic']:>9.3f}  {ic['gap']:>7.3f}  {n:>6}")

print()
print('Delta (B - A):')
for h, ic_a, ic_b in [('3d', ic_a_3d, ic_b_3d), ('5d', ic_a_5d, ic_b_5d)]:
    dg  = ic_b['ic_all'] - ic_a['ic_all']
    dcs = ic_b['cs_ic']  - ic_a['cs_ic']
    dgp = ic_b['gap']    - ic_a['gap']
    print(f"  {h}  Global IC {dg:+.3f}   CS IC {dcs:+.3f}   Gap {dgp:+.3f}")

## IC by fold: A vs B

In [ ]:
fig = plt.figure(figsize=(16, 12))
gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=0.45, wspace=0.35)
fig.suptitle(
    'RV Butterfly: Effect of Post-MPM Exclusion\n'
    'A = post-MPM excluded (main)   |   B = post-MPM included',
    fontsize=13, fontweight='bold'
)

def plot_ab_fold(ax, ic_a, ic_b, title):
    folds  = sorted(set(ic_a['ic_by_fold']) | set(ic_b['ic_by_fold']))
    a_vals = [ic_a['ic_by_fold'].get(f, np.nan) for f in folds]
    b_vals = [ic_b['ic_by_fold'].get(f, np.nan) for f in folds]
    x, w = np.arange(len(folds)), 0.35
    ax.bar(x - w/2, a_vals, w, label='A (excluded)', color='#1f77b4', alpha=0.85, edgecolor='white')
    ax.bar(x + w/2, b_vals, w, label='B (included)', color='#ff7f0e', alpha=0.85, edgecolor='white')
    ax.axhline(0, color='black', lw=0.8)
    ax.set_xticks(x); ax.set_xticklabels([str(f) for f in folds])
    ax.set_xlabel('Fold'); ax.set_ylabel('Global IC')
    ax.set_title(title, fontweight='bold')
    ax.legend(fontsize=9); ax.grid(True, axis='y', alpha=0.3)
    for i, (va, vb) in enumerate(zip(a_vals, b_vals)):
        if not np.isnan(va):
            ax.text(i - w/2, va + (0.01 if va >= 0 else -0.025), f'{va:.2f}',
                    ha='center', fontsize=7, color='#1f77b4')
        if not np.isnan(vb):
            ax.text(i + w/2, vb + (0.01 if vb >= 0 else -0.025), f'{vb:.2f}',
                    ha='center', fontsize=7, color='#ff7f0e')

plot_ab_fold(fig.add_subplot(gs[0, 0]), ic_a_3d, ic_b_3d,
             f"Global IC by Fold (3d)  |  A={ic_a_3d['ic_all']:.3f}  B={ic_b_3d['ic_all']:.3f}")
plot_ab_fold(fig.add_subplot(gs[0, 1]), ic_a_5d, ic_b_5d,
             f"Global IC by Fold (5d)  |  A={ic_a_5d['ic_all']:.3f}  B={ic_b_5d['ic_all']:.3f}")

# CS IC by fold (manual calculation)
def cs_ic_by_fold(results):
    out = {}
    for fold, grp in results.groupby('Fold'):
        boj = grp[grp['Meeting_Index'].isin(FLY_INDICES)]
        cs = []
        for _, dgrp in boj.groupby('Date'):
            if len(dgrp) >= 3:
                ic, _ = spearmanr(dgrp['Actual'], dgrp['Pred'])
                if not np.isnan(ic):
                    cs.append(ic)
        out[fold] = float(np.mean(cs)) if cs else np.nan
    return out

cs_a_3d = cs_ic_by_fold(res_a_3d)
cs_b_3d = cs_ic_by_fold(res_b_3d)
cs_a_5d = cs_ic_by_fold(res_a_5d)
cs_b_5d = cs_ic_by_fold(res_b_5d)

def plot_ab_cs_fold(ax, cs_a, cs_b, ic_a, ic_b, title):
    folds  = sorted(set(cs_a) | set(cs_b))
    a_vals = [cs_a.get(f, np.nan) for f in folds]
    b_vals = [cs_b.get(f, np.nan) for f in folds]
    x, w = np.arange(len(folds)), 0.35
    ax.bar(x - w/2, a_vals, w, label='A (excluded)', color='#1f77b4', alpha=0.85, edgecolor='white')
    ax.bar(x + w/2, b_vals, w, label='B (included)', color='#ff7f0e', alpha=0.85, edgecolor='white')
    ax.axhline(0, color='black', lw=0.8)
    ax.set_xticks(x); ax.set_xticklabels([str(f) for f in folds])
    ax.set_xlabel('Fold'); ax.set_ylabel('CS IC')
    ax.set_title(title, fontweight='bold')
    ax.legend(fontsize=9); ax.grid(True, axis='y', alpha=0.3)
    for i, (va, vb) in enumerate(zip(a_vals, b_vals)):
        if not np.isnan(va):
            ax.text(i - w/2, va + (0.01 if va >= 0 else -0.025), f'{va:.2f}',
                    ha='center', fontsize=7, color='#1f77b4')
        if not np.isnan(vb):
            ax.text(i + w/2, vb + (0.01 if vb >= 0 else -0.025), f'{vb:.2f}',
                    ha='center', fontsize=7, color='#ff7f0e')

plot_ab_cs_fold(fig.add_subplot(gs[1, 0]), cs_a_3d, cs_b_3d, ic_a_3d, ic_b_3d,
                f"CS IC by Fold (3d)  |  A={ic_a_3d['cs_ic']:.3f}  B={ic_b_3d['cs_ic']:.3f}")
plot_ab_cs_fold(fig.add_subplot(gs[1, 1]), cs_a_5d, cs_b_5d, ic_a_5d, ic_b_5d,
                f"CS IC by Fold (5d)  |  A={ic_a_5d['cs_ic']:.3f}  B={ic_b_5d['cs_ic']:.3f}")

plt.show()